# Notebook 2 : Pandas avancé

In [1]:
import pandas as pd

## Tennis

Nous considérons les données des résultats des matchs de tennis masculin des tournois de Roland Garros et Wimbledon en 2013. La liste des variables et leur signification se trouvent sur [cette page](https://archive.ics.uci.edu/dataset/300/tennis+major+tournament+match+statistics) dans la section *Additional Variable Information*.

1. Commencer par charger le jeu de données relatif au tournoi de Roland Garros dans un dataframe `rg` à partir du fichier `rolandgarros2013.csv`.

In [3]:
rg=pd.read_csv('data/rolandgarros2013.csv',sep=',')

rg.head(10)

,Player1,Player2,Round,Result,FNL.1,FNL.2,FSP.1,FSW.1,SSP.1,SSW.1,...,BPC.2,BPW.2,NPA.2,NPW.2,TPW.2,ST1.2,ST2.2,ST3.2,ST4.2,ST5.2
0,Pablo Carreno-Busta,Roger Federer,1,0,0,3,62,27,38,11,...,7,7,14,18,88,6,6,6.0,NaN,NaN
1,Somdev Devvarman,Daniel Munoz-De La Nava,1,1,3,0,62,54,38,22,...,1,16,22,25,106,3,3,5.0,NaN,NaN
2,Tobias Kamke,Paolo Lorenzi,1,1,3,2,62,53,38,15,...,10,18,19,27,139,3,3,6.0,6.0,3.0
3,Julien Benneteau,Ricardas Berankis,1,1,3,1,72,87,28,19,...,4,13,33,43,149,6,3,7.0,6.0,NaN
4,Lukas Lacko,Sam Querrey,1,0,0,3,52,31,48,22,...,4,7,12,13,93,6,6,6.0,NaN,NaN
5,Jan Hajek,Denis Kudla,1,1,3,1,70,58,30,18,...,1,7,6,9,93,2,7,0.0,4.0,NaN
6,Adrian Mannarino,Pablo Cuevas,1,0,2,3,63,71,37,38,...,6,20,14,22,175,6,2,6.0,5.0,7.0
7,Gilles Simon,Lleyton Hewitt,1,1,3,2,59,42,41,25,...,9,10,19,35,120,6,6,4.0,1.0,5.0
8,Philipp Petzschner,Marin Cilic,1,0,0,3,56,27,44,13,...,7,12,13,21,92,6,6,6.0,NaN,NaN
9,Radek Stepanek,Nick Kyrgios,1,0,0,3,63,62,37,29,...,1,3,9,20,130,7,7,7.0,NaN,NaN


2. Afficher les noms des demi-finalistes.

In [8]:
rg.groupby('Round').agg(nb_lignes=pd.NamedAgg(column="Player1", aggfunc="count")).sort_values(by=['nb_lignes'],ascending=False)

,nb_lignes
Round,
1,63
2,31
3,16
4,8
5,4
6,2
7,1


In [11]:
rg[rg['Round']==6]

,Player1,Player2,Round,Result,FNL.1,FNL.2,FSP.1,FSW.1,SSP.1,SSW.1,...,BPC.2,BPW.2,NPA.2,NPW.2,TPW.2,ST1.2,ST2.2,ST3.2,ST4.2,ST5.2
122,David Ferrer,Jo-Wilfried Tsonga,6,1,3,0,60,35,40,23,...,2,5,7,16,84,1,6,2.0,NaN,NaN
123,Novak Djokovic,Rafael Nadal,6,0,2,3,67,76,33,30,...,8,16,15,26,177,6,3,6.0,6.0,9.0


3. Calculer le nombre moyen d'aces par match dans le tournoi.

In [12]:
rg['Ace_tot']=rg['ACE.1']+rg['ACE.2']
print(rg['Ace_tot'].mean())

12.688


4. Combien y a-t-il eu d'aces par match en moyenne à chaque niveau du tournoi ?

In [13]:
print(
    rg.groupby('Round')
        .agg( # NamedAgg permet de nommer les agrégations
            nb_ace_moy=pd.NamedAgg(column="Ace_tot", aggfunc="mean")
        )
)

       nb_ace_moy
Round            
1       13.476190
2       13.193548
3       12.562500
4        9.125000
5        7.000000
6       10.000000
7        6.000000


5. Filtrer les matchs pour lesquels au moins une des variables `DBF.1` et `DBF.2` est manquante.

In [16]:
rg[rg['DBF.1'].isna()|rg['DBF.2'].isna()].filter(items=["Player1", "Player2",'DBF.1','DBF.2'])

,Player1,Player2,DBF.1,DBF.2
56,Simone Bolelli,Yen-Hsun Lu,NaN,NaN
63,Somdev Devvarman,Roger Federer,NaN,NaN


6. Remplacer les valeurs manquantes de `DBF.1` par zéro avec la méthode `loc`.

In [17]:
rg.loc[rg['DBF.1'].isna(), "DBF.1"] = 0
print(rg[rg['DBF.1'].isna()|rg['DBF.2'].isna()].filter(items=["Player1", "Player2",'DBF.1','DBF.2']))

             Player1        Player2  DBF.1  DBF.2
56    Simone Bolelli    Yen-Hsun Lu    0.0    NaN
63  Somdev Devvarman  Roger Federer    0.0    NaN


7. Remplacer les valeurs manquantes de `DBF.2` par zéro avec la méthode `fillna`.

In [18]:
rg['DBF.2'].fillna(0, inplace=True)

/tmp/ipykernel_42208/2457930233.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  rg['DBF.2'].fillna(0, inplace=True)


8. Extraire la liste des participants à partir des colonnes `Player1` et `Player2`. Une façon de faire consiste à utiliser `concat` et la méthode `drop_duplicates` pour obtenir le résultat sous la forme d'une série et de la convertir en dataframe avec la méthode `to_frame`.

In [24]:
player_list=pd.concat([rg['Player1'],rg['Player2']]).drop_duplicates().to_frame()
print(player_list)

                      0
0   Pablo Carreno-Busta
1      Somdev Devvarman
2          Tobias Kamke
3      Julien Benneteau
4           Lukas Lacko
..                  ...
56          Yen-Hsun Lu
59      Grigor Dimitrov
61          Guido Pella
62         David Goffin
87     Janko Tipsarevic

[127 rows x 1 columns]


9. Écrire une fonction `n_match` qui prend une chaîne de caractères `joueur` en entrée et retourne le nombre de matchs disputés par le joueur.

In [ ]:
def n_match(s):
    

10. Utiliser les deux question précédentes et la méthode `apply` pour compter le nombre de matchs que chaque participant a disputé et ordonner le résultat par ordre décroissant.

11. Charger maintenant le jeu de données relatif au tournoi de Wimbledon dans un dataframe `wb` à partir du fichier `wimbledon2013.csv`.

12. Ajouter une colonne `Tournoi` dans les dataframes `rg` et `wb` contenant respectivement les chaînes de caractères `"RG"` et `"WB"`.

13. Concaténer les deux dataframes dans un nouveau dataframe `tennis`.

14. Utiliser le dataframe `tennis` pour comparer le nombre moyen d'aces par match à chaque niveau du tournoi à Roland Garros et à Wimbledon. Afficher le résultat en format large.

15. Quelle différence y a-t-il dans le format des noms des joueurs entre les dataframes `rg` et `wb` ?

16. Construire un dataframe `rg_victoires` avec les trois colonnes suivantes pour le tournoi de Roland Garros :
- `joueur` : nom du joueur tel qu'il est donné dans `rg`,
- `nom_joueur` : nom de famille du joueur uniquement,
- `n_victoire` : nombre de matchs gagnés dans le tournoi.

17. Construire un dataframe `wb_victoires` avec les trois colonnes suivantes pour le tournoi de Wimbledon :
- `joueur` : nom du joueur tel qu'il est donné dans `wb`,
- `nom_joueur` : nom de famille du joueur uniquement,
- `n_victoire` : nombre de matchs gagnés dans le tournoi.

18. Faire une jointure entre `rg_victoires` et `wb_victoires` sur la colonne `nom_joueur` pour comparer le nombre de victoires par tournoi pour chaque joueur. Expliquer la différence de résultat selon que la jointure est à gauche, à droite, intérieure ou extérieure.